# Library Database Project

Name: Meziane Ouchene  
Class: CIS 3120 – Programming for Analytics  
Semester: Spring 2026  

## Introduction

In this project I created a small database for a public library using SQLite and Python. I worked with three CSV files which are Book, Member, and Loan. First, I downloaded the data from GitHub and saved it locally. Then I created the tables in the database and made sure the relationships between them were correct using primary keys and foreign keys.

This project helped me understand better how databases work and how Python can be used to manage and query data.

Imports:

In [39]:
import sqlite3
import csv
import urllib.request
import os

link my project with the RAW URLs:

In [40]:
BOOK_URL = "https://raw.githubusercontent.com/mezianeouchene18/CIS3120-Library_DataBase/refs/heads/main/Book.csv"
MEMBER_URL = "https://raw.githubusercontent.com/mezianeouchene18/CIS3120-Library_DataBase/refs/heads/main/Member.csv"
LOAN_URL = "https://raw.githubusercontent.com/mezianeouchene18/CIS3120-Library_DataBase/refs/heads/main/Loan.csv"

BOOK_PATH = "/content/Book.csv"
MEMBER_PATH = "/content/Member.csv"
LOAN_PATH = "/content/Loan.csv"

DB_PATH = "/content/library.db"

downloading the files:

In [41]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:
    urllib.request.urlretrieve(url, path)
    print("Downloaded:", path)

Downloaded: /content/Book.csv
Downloaded: /content/Member.csv
Downloaded: /content/Loan.csv


create and connecting the database:

In [42]:
import os

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print("Old database deleted.")
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

conn.execute("""
CREATE TABLE IF NOT EXISTS Book (
    callNo TEXT NOT NULL,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    PRIMARY KEY (callNo)
);
""")
conn.execute("""
CREATE TABLE IF NOT EXISTS Member (
    id INTEGER NOT NULL,
    firstname TEXT NOT NULL,
    lastName TEXT NOT NULL,
    PRIMARY KEY (id)
);
""")
conn.execute("""
CREATE TABLE IF NOT EXISTS Loan (
    callNo TEXT NOT NULL,
    id INTEGER NOT NULL,
    dateBorrowed TEXT NOT NULL,
    dateReturned TEXT,
    dateDue TEXT NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id) REFERENCES Member(id)
);
""")
conn.commit()

print("the tables are created successfully")

Old database deleted.
the tables are created successfully


loading the books:

In [43]:
book_data = []
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        book_data.append((row['callNo'], row['title'], row['author']))

try:
    conn.executemany(
        "INSERT INTO Book VALUES (?, ?, ?);",
        book_data
    )
    conn.commit()
    print("the books are loaded successfully")
except sqlite3.OperationalError as e:
    print(f"error to load books: {e}")
    conn.rollback() # Rollback in case of error
    print("the transaction didn't go through")
except Exception as e:
    print(f"an unexpected error occurred: {e}")
    conn.rollback()
    print("the transaction didn't go through")

the books are loaded successfully


loading the members:

In [44]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Member VALUES (?, ?, ?);",
            (int(row['id']), row['firstname'], row['lastName'])
        )

conn.commit()
print("the members are loaded. done succesfully")

the members are loaded. done succesfully


loading the loan:

In [45]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        date_returned = row['dateReturned'] if row['dateReturned'].strip() else None

        conn.execute("""
        INSERT INTO Loan VALUES (?, ?, ?, ?, ?);
        """, (
            row['callNo'],
            int(row['id']),
            row['dateBorrowed'],
            date_returned,
            row['dateDue']
        ))

conn.commit()
print("the loans are loaded succesfully")

the loans are loaded succesfully


In [46]:
print("Books:", conn.execute("SELECT COUNT(*) FROM Book").fetchone()[0])
print("Members:", conn.execute("SELECT COUNT(*) FROM Member").fetchone()[0])
print("Loans:", conn.execute("SELECT COUNT(*) FROM Loan").fetchone()[0])

Books: 11
Members: 4
Loans: 4
